In [ ]:
# Run this first. Packages are installed into the active Jupyter kernel.
import shutil
import subprocess
import sys

if sys.version_info < (3, 14):
    raise RuntimeError("qlang requires Python 3.14 or newer. Select a compatible kernel and rerun this cell.")

requirements = [
    "qlang @ git+https://github.com/asherhe/qlang.git",
    "ipywidgets",
    "matplotlib",
]

uv = shutil.which("uv")
command = (
    [uv, "pip", "install", "--python", sys.executable, "--upgrade", *requirements]
    if uv
    else [sys.executable, "-m", "pip", "install", "--upgrade", *requirements]
)
print("Installing into the active kernel:", sys.executable)
subprocess.check_call(command)


# qlang live schematic

This standalone notebook compiles symbolic predicates into Qiskit phase-oracle circuits. Run the first cell before using the dashboard. It installs packages into the Python environment behind the current Jupyter kernel. To keep them inside this repository's uv environment, launch Jupyter with that environment and choose `.venv\Scripts\python` as the kernel.


In [ ]:
import ast
import re

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

import qlang


## Live compiler

Edit any field below. Declarations and domains use one variable per line. Write `x = 3` for a three-bit unsigned integer or `flag = bit` for a Boolean; domains accept an inclusive range such as `x = 0..7` or a value list such as `x = 0, 2, 5`.


In [ ]:
expression_input = widgets.Textarea(
    value="x + y == 5",
    description="Expression",
    placeholder="(x + y == 5) & (flag | ~flag)",
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="72px"),
)
declarations_input = widgets.Textarea(
    value="x = 3\ny = 3",
    description="Variables",
    placeholder="x = 3\nflag = bit",
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="88px"),
)
domains_input = widgets.Textarea(
    value="x = 0..7\ny = 0..7",
    description="Domains",
    placeholder="x = 0..7\ny = 0, 2, 5",
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="88px"),
)
for control in (expression_input, declarations_input, domains_input):
    control.style.description_width = "90px"

status = widgets.HTML("<em>Waiting for the first compilation.</em>")
circuit_output = widgets.Output(
    layout=widgets.Layout(width="100%", overflow="auto", padding="8px")
)
with circuit_output:
    display(HTML("<em>The compiled circuit will appear here.</em>"))

circuit_panel = widgets.VBox(
    [widgets.HTML("<h4 style='margin: 0 0 8px'>Compiled phase oracle</h4>"), circuit_output],
    layout=widgets.Layout(width="100%", border="1px solid #cbd5e1", padding="12px"),
)
circuit_panel.add_class("qlang-circuit-panel")

dashboard_styles = widgets.HTML(
    """<style>
    .qlang-circuit-panel { transition: opacity 160ms ease; }
    .qlang-circuit-panel.qlang-dimmed { opacity: 0.35; }
    </style>"""
)
dashboard = widgets.VBox(
    [dashboard_styles, expression_input, declarations_input, domains_input, status, circuit_panel],
    layout=widgets.Layout(width="100%", max_width="none", gap="10px"),
)
display(dashboard)
